## 1. Loading the dataset into a pandas dataframe and seeing its dimensions


In [ ]:
import pandas as pd

df_ais = pd.read_csv('../data/raw/aisdk-2024-08-07.csv')
print(df_ais.shape)

The dataset contains 28,203,137 AIS records and 26 columns representing 24 hours of vessel activity 

## 2. Inspecting the dataset column structure

In [ ]:
print(df_ais.columns)

Index(['# Timestamp', 'Type of mobile', 'MMSI', 'Latitude', 'Longitude',
       'Navigational status', 'ROT', 'SOG', 'COG', 'Heading', 'IMO',
       'Callsign', 'Name', 'Ship type', 'Cargo type', 'Width', 'Length',
       'Type of position fixing device', 'Draught', 'Destination', 'ETA',
       'Data source type', 'A', 'B', 'C', 'D'],
      dtype='object')


The dataset contains 26 columns. The ones most relevant to this project are: 
- 'MMSI' -> unique vessel identifier
- 'Latitude' and 'Longitude' -> vessel position
- 'SOG' -> speed over ground in knots(kn)
- 'Ship type' -> vessel category
- 'Destination' -> reported destination(where available)
- 'ETA' -> reported ETA(where available)
- '# Timestamp' -> timestamp from the AIS basestation
- 'Type of mobile' -> type of target this message is received from (class A AIS Vessel, Class B AIS vessel, etc)
- 'Navigational status' -> navigational status from AIS message (where available), e.g.: 'Engaged in fishing', 'Under way using engine'
- 'Name' -> vessel name 

## 3. Exploring vessel records and target vessel types
Since the dataset has multiple million observations we will find:
- unique vessel MMSI's
- observations per unique MMSI
- unique vessel types(we target cargo, tanker and passenger)
- destination and ETA availability across unique MMSI's with our target type

In [ ]:
df_vessels = df_ais['MMSI'].nunique()
print(df_vessels)

df_obs = df_ais.groupby('MMSI').size()
print(df_obs)

There are 9102 unique MMSI's in the dataset.  
Some MMSI's, such as 148 and 3638, appear unusually short or non-standard. Since the analysis focuses on commercial cargo, tanker, and passenger vessels, Type of mobile = Class A is relevant, but unusual MMSI values still remain even within Class A records.

In [ ]:
vessel_obs = df_ais[df_ais['Type of mobile'] == 'Class A'].groupby('MMSI').size()
print(vessel_obs)

Some MMSI values appear unusually short or incomplete even among Class A records. These observations will be investigated and handled during the cleaning stage.

In [ ]:
df_types = df_ais.groupby('Ship type').size()
print(df_types)

The raw dataset contains:
- 5,628,117 cargo observations
- 2,647,622 tanker observations
- 3,566,724 passenger observations

In [ ]:
unique_mmsi_by_type = df_ais.groupby('Ship type')['MMSI'].nunique()
print(unique_mmsi_by_type)

Among the target vessel types the dataset contains:
- 694 cargo MMSI's
- 260 tanker MMSI's
- 435 passenger MMSI's
Keep in mind each MMSI does not necessarily represent an actual vessel

In [ ]:
df_target = df_ais[df_ais['Ship type'].isin(['Cargo', 'Tanker', 'Passenger'])]
eta_availability = df_target.groupby('Ship type')['ETA'].count()
dest_availability = df_target.groupby('Ship type')['Destination'].count()
print(eta_availability)
print(dest_availability)

ETA availability:
- 5,522,543 cargo observations
- 2,793,477 passenger observations
- 2,627,667 tanker observations

Destination availability:
- 5,628,117 cargo observations
- 3,566,724 passenger observations
- 2,647,622 tanker observations

Sidenote: non-null observations do not necessarily count as valid ones